# Music Annotation Project V9.0B
## Frozen V8.8B Transfer to the Notebook 06 to 07 Human-Reviewed Mask Population

**Runtime: CPU. Run all in a fresh Colab runtime.**

### Hypothesis
When the **exact operational REMOVE-only reconstruction from Notebook 53R3** is applied to the 151 resolved Notebook 06 to 07 human-reviewed mask lineages, correction safety and clean-page performance transfer to that operational HITL population. Remaining failures are associated with measurable human mask-edit characteristics.

### Scope correction
Notebook 53R3 is not a standalone model runner. Notebook 53R3 consumes durable upstream Notebook 45 regional M3 masks, Notebook 46 base/protected pages and Notebook 50 decision maps. This notebook therefore:

1. resolves the 151 V9.0A HITL lineages;
2. resolves the exact Notebook 45, 46R6 and 50 runs;
3. proves that every test page has the required upstream V8.8 artefacts;
4. executes the Notebook 53R3 **OPERATIONAL_REMOVE_ONLY** cells directly in this notebook;
5. locks outputs before clean truth is scored;
6. tests whether draft-to-reviewed mask edits predict remaining failures.

If the full 151 pages do not have Notebook 45/46/50 artefacts, the notebook stops with a precise coverage table. It does not substitute `geometry_t80`, raw Telea, or a fictional Python runner for V8.8B.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-learn statsmodels matplotlib

## 1. Configuration and fail-closed helpers

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,math,shutil,zipfile,sys,platform,gc
import cv2,numpy as np,pandas as pd,matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold,cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score,average_precision_score,brier_score_loss
from sklearn.inspection import permutation_importance
import statsmodels.api as sm

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V90A_ROOT=ROOT/'observed_print_classifier_v9_0a_mask_authority_transfer'
N45_ROOT=ROOT/'observed_print_classifier_v8_6_notebook45'
N46_ROOT=ROOT/'observed_print_classifier_v8_6_notebook46'
N50_ROOT=ROOT/'observable_residual_mask_m5_v8_7'
N53_ROOT=ROOT/'observed_print_classifier_v8_8_notebook53R3'
CLEAN_DIR=ROOT/'source'/'clean'
DRAFT_MASK_DIR=ROOT/'human_in_the_middle'/'draft_masks'
REVIEWED_MASK_DIR=ROOT/'masks'/'corrected'
OUT_ROOT=ROOT/'observed_print_classifier_v9_0b_hitl_transfer'
V90A_RUN_OVERRIDE=None; N45_RUN_OVERRIDE=None; N46_RUN_OVERRIDE=None; N50_RUN_OVERRIDE=None; N53_RUN_OVERRIDE=None
EXPECTED_PAGES=151; INPAINT_RADIUS=3; SAFE_REMOVE_CODE=1; SAFE_PROTECT_CODE=2; MIN_COMPONENT_AREA=20; BOOTSTRAP_RESAMPLES=10000; SEED=42
STRUCTURAL_GATES={'minimum_ink_recall':.985,'minimum_ink_f1':.95,'minimum_edge_f1':.90,'maximum_removed_print_fraction':.001,'maximum_component_mae':16.0}
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v9_0b_53R3_hitl_transfer'
OUT=OUT_ROOT/RUN; INPUTS=OUT/'01_inputs'; UPSTREAM=OUT/'02_upstream'; PAGES=OUT/'03_pages'; LOCKS=OUT/'04_lock'; TABLES=OUT/'05_tables'; VISUAL=OUT/'06_visual'; AUDIT=OUT/'07_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,UPSTREAM,PAGES,LOCKS,TABLES,VISUAL,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

def require(ok,msg,detail=None):
 if not ok:
  (AUDIT/'fatal_failure.json').write_text(json.dumps({'utc':datetime.now(timezone.utc).isoformat(),'message':str(msg),'detail':detail},indent=2,default=str))
  raise RuntimeError(msg)
def newest(root,required):
 q=[]
 if Path(root).exists():
  for p in Path(root).iterdir():
   if p.is_dir() and all((p/r).exists() for r in required):q.append(p)
 return max(q,key=lambda p:p.stat().st_mtime) if q else None
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def array_hash(a):
 a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def atomic(path,value):
 tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str));tmp.replace(path)
def imap(folder):
 ex={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'};return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in ex} if Path(folder).exists() else {}
def wilson_upper(e,n,z=1.96):
 if not n:return np.nan
 p=e/n;d=1+z*z/n;c=p+z*z/(2*n);r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n));return (c+r)/d
print('Output:',OUT)

## 2. Resolve the completed V9.0A roster and exact V8.8B records

In [ ]:
V90A=Path(V90A_RUN_OVERRIDE) if V90A_RUN_OVERRIDE else newest(V90A_ROOT,['audit/hitl_input_resolution.csv','audit/hitl_resolution_summary.json'])
N45=Path(N45_RUN_OVERRIDE) if N45_RUN_OVERRIDE else newest(N45_ROOT,['mask_lock.json','source_record.json','tables/region_manifest.parquet'])
N46=Path(N46_RUN_OVERRIDE) if N46_RUN_OVERRIDE else newest(N46_ROOT,['source_record.json','audit/output_completion.json'])
N50=Path(N50_RUN_OVERRIDE) if N50_RUN_OVERRIDE else newest(N50_ROOT,['notebook50_verdict.json','tables/m5_map_inventory.csv'])
N53=Path(N53_RUN_OVERRIDE) if N53_RUN_OVERRIDE else newest(N53_ROOT,['stage_f_verdict.json','experiment_identity.json','tables/page_equivalence.csv'])
for name,p in [('V9.0A',V90A),('Notebook45',N45),('Notebook46',N46),('Notebook50',N50),('Notebook53R3',N53)]:require(p is not None,f'No complete {name} run found.')
identity53=json.loads((N53/'experiment_identity.json').read_text()); verdict53=json.loads((N53/'stage_f_verdict.json').read_text())
require(identity53.get('version')=='53R3-v8.8b-notebook51R2-regression-compatibility','Unexpected Notebook53R3 identity.',identity53)
require(verdict53.get('stage_F_corrected_page_exact') is True,'Notebook53R3 Stage F did not pass.',verdict53)
require(verdict53.get('safe_protect_operationally_authorised') is False,'Unexpected SAFE_PROTECT authority state.')
record={'v9_0a':str(V90A),'notebook45':str(N45),'notebook46':str(N46),'notebook50':str(N50),'notebook53R3':str(N53),'notebook53R3_mode_reference':identity53.get('mode'),'v9_0b_execution_mode':'OPERATIONAL_REMOVE_ONLY','stage_f_reference_passed':True}
atomic(UPSTREAM/'resolved_runs.json',record);print(json.dumps(record,indent=2))

## 3. Resolve the 151 reviewed-mask lineages and audit upstream V8.8 coverage

In [ ]:
resolution=pd.read_csv(V90A/'audit/hitl_input_resolution.csv',dtype=str,keep_default_na=False)
need={'page','observed_path','raw_yolo_mask_path','reviewed_mask_path'}
require(need.issubset(resolution.columns),f'V9.0A resolution missing {sorted(need-set(resolution.columns))}')
resolution['page']=resolution.page.astype(str);require(not resolution.page.duplicated().any(),'Duplicate pages in V9.0A resolution.')
require(len(resolution)==EXPECTED_PAGES,f'Expected {EXPECTED_PAGES} resolved V9.0A pages, found {len(resolution)}.')
regions=pd.read_parquet(N45/'tables/region_manifest.parquet');regions['page']=regions.page.astype(str);regions['region_id']=regions.region_id.astype(str)
map_inventory=pd.read_csv(N50/'tables/m5_map_inventory.csv',dtype=str,keep_default_na=False);map_inventory['page']=map_inventory.page.astype(str)
source45=json.loads((N45/'source_record.json').read_text());source_v82=Path(source45['source_v82'])
rows=[]
for _,r in resolution.iterrows():
 page=r.page; n45_regions=int((regions.page==page).sum()); n50_rows=int((map_inventory.page==page).sum())
 observed=Path(r.observed_path); draft=Path(r.raw_yolo_mask_path); reviewed=Path(r.reviewed_mask_path); clean=imap(CLEAN_DIR).get(page)
 base_candidates=[N46/'whole_page_candidates'/page/'v7_8.png',source_v82/'whole_pages'/page/'v7_8.png'];base=next((p for p in base_candidates if p.exists()),None)
 protected=N46/'whole_page_candidates'/page/'protected_print.png'
 rows.append({'page':page,'observed_path':str(observed),'draft_mask_path':str(draft),'reviewed_mask_path':str(reviewed),'clean_path':str(clean) if clean else '', 'n45_regions':n45_regions,'n50_rows':n50_rows,'base_path':str(base) if base else '','protected_path':str(protected),'observed_exists':observed.exists(),'draft_exists':draft.exists(),'reviewed_exists':reviewed.exists(),'clean_exists':clean is not None,'base_exists':base is not None,'protected_exists':protected.exists(),'upstream_complete':bool(n45_regions>0 and n50_rows==1 and base is not None and protected.exists())})
coverage=pd.DataFrame(rows);coverage.to_csv(AUDIT/'upstream_coverage_by_page.csv',index=False)
summary={'resolved_hitl_pages':len(coverage),'upstream_complete_pages':int(coverage.upstream_complete.sum()),'missing_upstream_pages':coverage.loc[~coverage.upstream_complete,'page'].tolist()}
atomic(AUDIT/'upstream_coverage_summary.json',summary);print(json.dumps(summary,indent=2))
require(coverage.upstream_complete.all(),'The 151-page HITL cohort does not yet have complete Notebook45/46/50 artefacts. See upstream_coverage_by_page.csv. No substitute implementation is permitted.')
roster=coverage.copy();roster.to_csv(INPUTS/'v9_0b_roster.csv',index=False)

## 4. Human mask-edit characteristics, computed before truth

In [ ]:
def components(m,min_area=1):
 n,_,s,_=cv2.connectedComponentsWithStats(m.astype(np.uint8),8);return int(sum(s[i,4]>=min_area for i in range(1,n)))
def perimeter(m):
 c,_=cv2.findContours(m.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_NONE);return float(sum(cv2.arcLength(x,True) for x in c))
def edit_features(d,c):
 add=c&~d;erase=d&~c;inter=d&c;union=d|c;xor=add|erase;depth=cv2.distanceTransform(c.astype(np.uint8),cv2.DIST_L2,5)
 return {'draft_pixels':int(d.sum()),'reviewed_pixels':int(c.sum()),'added_pixels':int(add.sum()),'erased_pixels':int(erase.sum()),'xor_pixels':int(xor.sum()),'draft_reviewed_iou':float(inter.sum()/max(union.sum(),1)),'addition_fraction_final':float(add.sum()/max(c.sum(),1)),'erasure_fraction_draft':float(erase.sum()/max(d.sum(),1)),'net_area_change_fraction':float((c.sum()-d.sum())/max(d.sum(),1)),'draft_components':components(d,MIN_COMPONENT_AREA),'reviewed_components':components(c,MIN_COMPONENT_AREA),'component_count_change':components(c,MIN_COMPONENT_AREA)-components(d,MIN_COMPONENT_AREA),'edit_fragmentation':components(xor,1),'perimeter_change_fraction':float((perimeter(c)-perimeter(d))/max(perimeter(d),1)),'maximum_reviewed_mask_depth':float(depth[c].max()) if c.any() else 0.0,'mean_reviewed_mask_depth':float(depth[c].mean()) if c.any() else 0.0}
features=[]
for _,r in roster.iterrows():
 d=cv2.imread(r.draft_mask_path,0);c=cv2.imread(r.reviewed_mask_path,0);require(d is not None and c is not None and d.shape==c.shape,f'{r.page}: draft/review mask mismatch')
 q={'page':r.page};q.update(edit_features(d>0,c>0));features.append(q)
features=pd.DataFrame(features);features.to_csv(TABLES/'mask_edit_features_locked_candidate.csv',index=False)

## 5. Exact Notebook 53R3 operational REMOVE-only mask decoding

In [ ]:
def union_arm(page,arm='M3'):
 arr=None
 for _,r in regions[regions.page==page].iterrows():
  p=N45/'masks'/page/str(r.region_id).replace(':','_')/f'{arm}.png';m=cv2.imread(str(p),0);require(m is not None,f'Missing {p}');b=m>0;arr=b.copy() if arr is None else arr|b
 require(arr is not None,f'{page}: no {arm} masks');return arr
def decision_maps(page,shape):
 q=map_inventory[map_inventory.page==page];require(len(q)==1,f'{page}: expected one Notebook50 decision row, found {len(q)}');p=Path(q.iloc[0].file)
 if not p.exists():
  hits=[x for x in N50.rglob(p.name) if x.is_file()];require(len(hits)==1,f'{page}: cannot uniquely resolve decision NPZ');p=hits[0]
 with np.load(p,allow_pickle=False) as z:require('m5_decision_map' in z.files,f'{p}: missing m5_decision_map');small=np.asarray(z['m5_decision_map'])
 require(small.ndim==2 and small.shape[0]<=shape[0] and small.shape[1]<=shape[1],f'{page}: decision canvas shape invalid');decision=np.zeros(shape,dtype=small.dtype);decision[:small.shape[0],:small.shape[1]]=small
 return decision==SAFE_REMOVE_CODE,decision==SAFE_PROTECT_CODE,p,small.shape
def reconstruct_telea_exact(obs,mask):
 m=mask.astype(np.uint8)*255
 if not m.any():return obs.copy()
 cand=cv2.inpaint(obs,m,INPAINT_RADIUS,cv2.INPAINT_TELEA);out=obs.copy();out[m>0]=cand[m>0];require(np.array_equal(out[m==0],obs[m==0]),'Telea outside-mask drift');return out

## 6. Execute V8.8B operational REMOVE-only and release page memory immediately

In [ ]:
page_rows=[];region_rows=[]
for _,r in roster.iterrows():
 page=r.page;obs=cv2.imread(r.observed_path,0);base=cv2.imread(r.base_path,0);protected=cv2.imread(r.protected_path,0);m3=union_arm(page,'M3')
 require(obs is not None and base is not None and protected is not None and obs.shape==base.shape==protected.shape==m3.shape,f'{page}: input shape mismatch')
 sr,sp,dp,canvas=decision_maps(page,m3.shape);require(not np.any(sr&~m3),f'{page}: SAFE_REMOVE outside M3');require(not np.any(sp&~m3),f'{page}: SAFE_PROTECT outside M3');require(not np.any(sr&sp),f'{page}: decision overlap')
 m5=m3&~sr # exact OPERATIONAL_REMOVE_ONLY contract
 candidate=base.copy();candidate[protected>0]=obs[protected>0];regional_union=np.zeros_like(m5)
 for _,rr in regions[regions.page==page].iterrows():
  rid=str(rr.region_id);mp=N45/'masks'/page/rid.replace(':','_')/'M3.png';rm=cv2.imread(str(mp),0);require(rm is not None and rm.shape==obs.shape,f'{page}/{rid}: bad M3');regional=m5&(rm>0);regional_union|=regional
  if regional.any():recon=reconstruct_telea_exact(obs,regional);candidate[regional]=recon[regional];del recon
  region_rows.append({'page':page,'region_id':rid,'regional_M3_pixels':int((rm>0).sum()),'regional_M5_pixels':int(regional.sum()),'regional_mask_sha256':array_hash(regional.astype(np.uint8))})
 require(np.array_equal(regional_union,m5),f'{page}: regional union differs from M5');require(np.array_equal(candidate[~m3],base[~m3]),f'{page}: outside-M3 base drift')
 d=PAGES/page;d.mkdir(parents=True,exist_ok=False)
 for name,a in [('observed',obs),('base_v7_8',base),('m3',m3.astype(np.uint8)*255),('safe_remove',sr.astype(np.uint8)*255),('safe_protect_decision_audit_only',sp.astype(np.uint8)*255),('m5_remove',m5.astype(np.uint8)*255),('corrected',candidate)]:require(cv2.imwrite(str(d/f'{name}.png'),a),f'{page}: failed write {name}')
 page_rows.append({'page':page,'M3_pixels':int(m3.sum()),'SAFE_REMOVE_pixels':int(sr.sum()),'SAFE_PROTECT_decision_pixels_audit_only':int(sp.sum()),'M5_REMOVE_pixels':int(m5.sum()),'outside_M3_base_drift':0,'observed_sha256':array_hash(obs),'base_sha256':array_hash(base),'corrected_sha256':array_hash(candidate),'decision_npz':str(dp),'decision_npz_sha256':sha256_file(dp),'decision_canvas_shape':str(tuple(canvas))})
 del obs,base,protected,m3,sr,sp,m5,candidate,regional_union;gc.collect()
page_outputs=pd.DataFrame(page_rows);regions_out=pd.DataFrame(region_rows);page_outputs.to_csv(AUDIT/'page_output_inventory.csv',index=False);regions_out.to_csv(TABLES/'regional_masks.csv',index=False)
print('Built pages:',len(page_outputs))

## 7. Prediction and predictor lock before clean truth

In [ ]:
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0B-53R3-operational-remove-only','source_notebook':'53R3_V8_8B_Notebook51R2_Exact_Regression_Compatibility','execution_mode':'OPERATIONAL_REMOVE_ONLY','safe_protect_operationalised':False,'roster_sha256':sha256_file(INPUTS/'v9_0b_roster.csv'),'mask_edit_features_sha256':sha256_file(TABLES/'mask_edit_features_locked_candidate.csv'),'page_output_inventory_sha256':sha256_file(AUDIT/'page_output_inventory.csv'),'regional_masks_sha256':sha256_file(TABLES/'regional_masks.csv'),'page_files':{p:{x.name:sha256_file(x) for x in sorted((PAGES/p).iterdir()) if x.is_file()} for p in sorted(roster.page)}}
lock['lock_id']=hashlib.sha256(json.dumps(lock,sort_keys=True).encode()).hexdigest();atomic(LOCKS/'prediction_and_output_lock.json',lock);print('LOCK:',lock['lock_id'])

## 8. Post-lock clean-truth scoring

In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def tolerant(ref,pred,region):
 ref=ref&region;pred=pred&region
 if not ref.any():return np.nan,np.nan,np.nan
 k=np.ones((3,3),np.uint8);pr=(pred&(cv2.dilate(ref.astype(np.uint8),k)>0)).sum()/max(pred.sum(),1);re=(ref&(cv2.dilate(pred.astype(np.uint8),k)>0)).sum()/ref.sum();return float(pr),float(re),float(2*pr*re/max(pr+re,1e-12))
def score(clean,out,region,remove):
 ci,oi=printed_ink(clean),printed_ink(out);ip,ir,if1=tolerant(ci,oi,region);ep,er,ef1=tolerant(edges(clean),edges(out),region);diff=np.abs(clean.astype(np.float32)-out.astype(np.float32));mae=float(diff[region].mean()) if region.any() else np.nan;removed=float((remove&ci&region).sum()/max(region.sum(),1));checks={'ink_recall':not np.isfinite(ir) or ir>=STRUCTURAL_GATES['minimum_ink_recall'],'ink_f1':not np.isfinite(if1) or if1>=STRUCTURAL_GATES['minimum_ink_f1'],'edge_f1':not np.isfinite(ef1) or ef1>=STRUCTURAL_GATES['minimum_edge_f1'],'removed_print':removed<=STRUCTURAL_GATES['maximum_removed_print_fraction'],'component_mae':mae<=STRUCTURAL_GATES['maximum_component_mae']};return {'mae':mae,'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'removed_clean_print_fraction':removed,'structural_safe':all(checks.values()),'failure_reasons':'|'.join(k for k,v in checks.items() if not v)}
page_scores=[];component_scores=[]
for _,r in roster.iterrows():
 page=r.page;clean=cv2.imread(r.clean_path,0);out=cv2.imread(str(PAGES/page/'corrected.png'),0);m3=cv2.imread(str(PAGES/page/'m3.png'),0)>0;remove=cv2.imread(str(PAGES/page/'safe_remove.png'),0)>0
 require(clean is not None and out is not None and clean.shape==out.shape==m3.shape,f'{page}: truth shape mismatch');q=score(clean,out,m3,remove);q.update({'page':page,'M3_pixels':int(m3.sum())});page_scores.append(q)
 n,lab,st,_=cv2.connectedComponentsWithStats(m3.astype(np.uint8),8)
 for cid in range(1,n):
  if st[cid,4]<MIN_COMPONENT_AREA:continue
  q=score(clean,out,lab==cid,remove);q.update({'page':page,'component':cid,'x':int(st[cid,0]),'y':int(st[cid,1]),'w':int(st[cid,2]),'h':int(st[cid,3]),'area':int(st[cid,4])});component_scores.append(q)
 del clean,out,m3,remove,lab;gc.collect()
page_scores=pd.DataFrame(page_scores);component_scores=pd.DataFrame(component_scores);page_scores.to_csv(TABLES/'page_transfer_outcomes.csv',index=False);component_scores.to_parquet(TABLES/'component_transfer_outcomes.parquet',index=False)

## 9. Transfer summary and clustered uncertainty

In [ ]:
unsafe=int((~page_scores.structural_safe).sum());n=len(page_scores);rng=np.random.default_rng(SEED);boots=[];ids=page_scores.page.to_numpy();ix=page_scores.set_index('page')
for _ in range(BOOTSTRAP_RESAMPLES):
 q=ix.loc[rng.choice(ids,len(ids),replace=True)];boots.append([q.ink_recall.mean(),q.ink_f1.mean(),q.edge_f1.mean(),q.mae.mean(),(~q.structural_safe).mean()])
b=np.asarray(boots,float);summary={'pages':n,'unsafe_pages':unsafe,'unsafe_rate':unsafe/max(n,1),'unsafe_wilson95_upper':wilson_upper(unsafe,n),'mean_ink_recall':float(page_scores.ink_recall.mean()),'mean_ink_f1':float(page_scores.ink_f1.mean()),'mean_edge_f1':float(page_scores.edge_f1.mean()),'mean_mae':float(page_scores.mae.mean()),'bootstrap95':{'ink_recall':[float(np.nanquantile(b[:,0],.025)),float(np.nanquantile(b[:,0],.975))],'ink_f1':[float(np.nanquantile(b[:,1],.025)),float(np.nanquantile(b[:,1],.975))],'edge_f1':[float(np.nanquantile(b[:,2],.025)),float(np.nanquantile(b[:,2],.975))],'mae':[float(np.nanquantile(b[:,3],.025)),float(np.nanquantile(b[:,3],.975))],'unsafe_rate':[float(np.nanquantile(b[:,4],.025)),float(np.nanquantile(b[:,4],.975))]}}
atomic(OUT/'transfer_summary.json',summary);print(json.dumps(summary,indent=2))

## 10. Do mask-edit characteristics predict failures?

In [ ]:
data=features.merge(page_scores,on='page',validate='one_to_one');data['failure']=~data.structural_safe;data.to_csv(TABLES/'page_analysis_dataset.csv',index=False)
predictors=['draft_pixels','reviewed_pixels','added_pixels','erased_pixels','xor_pixels','draft_reviewed_iou','addition_fraction_final','erasure_fraction_draft','net_area_change_fraction','draft_components','reviewed_components','component_count_change','edit_fragmentation','perimeter_change_fraction','maximum_reviewed_mask_depth','mean_reviewed_mask_depth']
X=data[predictors].replace([np.inf,-np.inf],np.nan);y=data.failure.astype(int);uni=[]
for p in predictors:
 q=pd.DataFrame({'x':pd.to_numeric(X[p],errors='coerce'),'y':y}).dropna();iq=q.x.quantile(.75)-q.x.quantile(.25)
 if len(q)<20 or q.y.nunique()<2 or iq==0:uni.append({'predictor':p,'odds_ratio_per_iqr':np.nan,'ci95_low':np.nan,'ci95_high':np.nan,'p_value':np.nan});continue
 try:
  z=(q.x-q.x.median())/iq;fit=sm.Logit(q.y,sm.add_constant(z)).fit(disp=False);bb,se=fit.params.iloc[1],fit.bse.iloc[1];uni.append({'predictor':p,'odds_ratio_per_iqr':float(np.exp(bb)),'ci95_low':float(np.exp(bb-1.96*se)),'ci95_high':float(np.exp(bb+1.96*se)),'p_value':float(fit.pvalues.iloc[1])})
 except Exception as e:uni.append({'predictor':p,'error':str(e)})
pd.DataFrame(uni).to_csv(TABLES/'univariable_mask_edit_predictors.csv',index=False)
model_rows=[];oof=[];importance=[]
if y.nunique()>=2:
 cv=GroupKFold(n_splits=min(10,len(data)))
 models={'logistic':Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=5000,random_state=SEED))]),'forest':Pipeline([('impute',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=1000,min_samples_leaf=5,class_weight='balanced_subsample',random_state=SEED,n_jobs=-1))])}
 for name,model in models.items():
  pr=cross_val_predict(model,X,y,groups=data.page,cv=cv,method='predict_proba')[:,1];model_rows.append({'model':name,'roc_auc':roc_auc_score(y,pr),'average_precision':average_precision_score(y,pr),'brier':brier_score_loss(y,pr),'prevalence':float(y.mean())});oof.extend({'page':data.iloc[i].page,'model':name,'truth':int(y.iloc[i]),'probability':float(pr[i])} for i in range(len(y)));model.fit(X,y);pi=permutation_importance(model,X,y,n_repeats=50,random_state=SEED,scoring='average_precision');importance.extend({'model':name,'predictor':p,'importance_mean':float(a),'importance_std':float(s)} for p,a,s in zip(predictors,pi.importances_mean,pi.importances_std))
pd.DataFrame(model_rows).to_csv(TABLES/'page_grouped_failure_model_summary.csv',index=False);pd.DataFrame(oof).to_csv(TABLES/'page_grouped_failure_oof_predictions.csv',index=False);pd.DataFrame(importance).to_csv(TABLES/'failure_predictor_importance.csv',index=False)

## 11. Failure visuals, verdict and upload package

In [ ]:
rank=component_scores.assign(severity=(~component_scores.structural_safe).astype(int)*1000+component_scores.mae).sort_values('severity',ascending=False).head(30)
for j,(_,r) in enumerate(rank.iterrows(),1):
 page=r.page;obs=cv2.imread(str(PAGES/page/'observed.png'),0);out=cv2.imread(str(PAGES/page/'corrected.png'),0);clean=cv2.imread(roster.set_index('page').loc[page,'clean_path'],0);draft=cv2.imread(roster.set_index('page').loc[page,'draft_mask_path'],0);review=cv2.imread(roster.set_index('page').loc[page,'reviewed_mask_path'],0);m=50;x0=max(0,int(r.x)-m);y0=max(0,int(r.y)-m);x1=min(obs.shape[1],int(r.x+r.w)+m);y1=min(obs.shape[0],int(r.y+r.h)+m);ims=[obs[y0:y1,x0:x1],draft[y0:y1,x0:x1],review[y0:y1,x0:x1],out[y0:y1,x0:x1],clean[y0:y1,x0:x1]];fig,ax=plt.subplots(1,5,figsize=(20,4))
 for a,im,t in zip(ax,ims,['observed','Notebook06 draft','Notebook07 reviewed','V8.8B corrected','clean truth']):a.imshow(im,cmap='gray',vmin=0,vmax=255);a.set_title(t);a.axis('off')
 fig.suptitle(f'{page} component {int(r.component)} | safe={r.structural_safe} | {r.failure_reasons} | MAE={r.mae:.2f}');plt.tight_layout();plt.savefig(VISUAL/f'{j:03d}_{page}_c{int(r.component)}.png',dpi=150,bbox_inches='tight');plt.close(fig)
models=pd.read_csv(TABLES/'page_grouped_failure_model_summary.csv') if (TABLES/'page_grouped_failure_model_summary.csv').exists() else pd.DataFrame();predictive='PREDICTIVE_SIGNAL_DETECTED' if len(models) and models.average_precision.max()>models.prevalence.max()+.05 else 'NO_CLEAR_OUT_OF_FOLD_SIGNAL'
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'9.0B-53R3-operational-remove-only-HITL-transfer','verdict':'VALID_EVIDENCE_PACKAGE','pages':n,'unsafe_pages':unsafe,'unsafe_wilson95_upper':summary['unsafe_wilson95_upper'],'mask_edit_predictor_result':predictive,'execution_mode':'OPERATIONAL_REMOVE_ONLY','safe_protect_operationally_authorised':False,'raw_yolo_mask_authority_tested':False,'production_authority':False,'lock_id':lock['lock_id']};atomic(OUT/'v9_0b_verdict.json',verdict)
(OUT/'executive_summary.txt').write_text('
'.join(['V9.0B FROZEN V8.8B TRANSFER TO HUMAN-REVIEWED HITL MASKS',f'Pages: {n}',f'Unsafe pages: {unsafe}',f'Unsafe Wilson 95% upper: {summary["unsafe_wilson95_upper"]:.6f}',f'Mask-edit predictor result: {predictive}','Execution: exact Notebook53R3 operational REMOVE-only cells.','SAFE_PROTECT is audit-only and not operationally authorised.','Raw YOLO mask authority was not tested.']))
required=[OUT/'executive_summary.txt',OUT/'v9_0b_verdict.json',OUT/'transfer_summary.json',LOCKS/'prediction_and_output_lock.json',AUDIT/'upstream_coverage_by_page.csv',AUDIT/'page_output_inventory.csv',TABLES/'page_analysis_dataset.csv',TABLES/'univariable_mask_edit_predictors.csv',TABLES/'page_grouped_failure_model_summary.csv',TABLES/'failure_predictor_importance.csv']
for p in required:require(p.exists(),f'Missing upload artefact: {p}')
rows=[]
for src in required+sorted(VISUAL.glob('*.png'))[:30]:
 cat='visuals' if src.suffix=='.png' else 'tables' if src.suffix=='.csv' else 'audit';dst=UPLOAD/cat/src.name;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst);rows.append({'relative_path':str(dst.relative_to(UPLOAD)),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'README.txt').write_text('V9.0B exact Notebook53R3 operational REMOVE-only transfer experiment on 151 Notebook06 to 07 reviewed-mask lineages. No raw mask authority or production promotion.');z=OUT/'60_V9_0B_53R3_HITL_Transfer_upload_bundle.zip'
with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as a:
 for p in UPLOAD.rglob('*'):
  if p.is_file():a.write(p,p.relative_to(UPLOAD.parent))
print(json.dumps(verdict,indent=2));print('Bundle:',z)

## Interpretation

A completed run answers the transfer question only because every one of the 151 reviewed-mask pages was first shown to possess the complete Notebook 45, 46 and 50 upstream V8.8 state required by Notebook 53R3. If the coverage gate fails, the result is **missing upstream V8.8 inference**, not failed transfer and not permission to replace V8.8 with V7.8 or plain Telea.

**Next question:** Which upstream notebook should be commissioned to generate Notebook 45/46/50 artefacts for any reviewed-mask pages missing from the 151-page transfer roster?